In [2]:
import duckdb
import pandas as pd
from scipy import stats

con = duckdb.connect("../Data/processed/project.duckdb")

trades_with_trader_df = con.execute("SELECT * FROM trades_with_trader WHERE traderId IS NOT NULL").fetchdf()
rebuilt_account_summary = pd.read_csv("../Data/processed/rebuilt_account_summary.csv")


DATA : The data being used is essentially the masterdataset stored as a table in project.duckdb as trades_with traders. 

- We are treating email as a primary key. As in, an emailID means a unique trader. According to PDPA laws, we cannot keep the email, tele ID column and therefore we created a "traderID" column, which is a unique identifier made by hashing the emailID (creating a unique serial number by hashing the email). 
- We added a campaignID column also, which just points to which campaign a user was a part of. We added this to both trades and users dataset files
- Then we joined these two to create trades_with_trader, which includes all columns of trades + traderID. We did this by joining users and trades on the condition that the campaign ID and account ID both were same. (in a given campaign, the accountID will belong to one person only, even though across campaigns the accountID is recycled)

# Data quality checks

In [3]:
print("\nTotal trades:", len(trades_with_trader_df))
print("Distinct traders:", trades_with_trader_df["traderId"].nunique())
print("\nNulls per column:\n", trades_with_trader_df.isna().sum())
print("\nDuplicate rows:", trades_with_trader_df.duplicated().sum())



Total trades: 46376
Distinct traders: 3550

Nulls per column:
 accountId            0
closeTradeId         0
positionId           0
closeOrderId         0
openOrderId          0
durationSec          0
openDateTime         0
closeDateTime        0
profit               0
reverseProfit        0
netProfit            0
commission           0
amount               0
openPrice            0
closePrice           0
slPrice          24546
tpPrice          20886
side                 0
userGroupId          0
filename             0
campaignId           0
has_SL               0
has_TP               0
traderId             0
dtype: int64

Duplicate rows: 0


In [4]:
#Range sanity check
print(trades_with_trader_df[["durationSec", "amount", "netProfit"]].describe())

        durationSec        amount     netProfit
count  46376.000000  46376.000000  46376.000000
mean    1413.167177      0.183879     -6.662067
std     3925.868104      0.144554    106.069356
min        0.000000      0.010000   -944.100000
25%       78.000000      0.090000    -51.700000
50%      272.000000      0.120000     -0.640000
75%      995.000000      0.250000     45.400000
max    79210.000000      0.630000    680.640000


# Trader level profiling

In [5]:
trades_per_trader = trades_with_trader_df.groupby("traderId").size()
print("\nTrades per trader — mean:", trades_per_trader.mean().round(1),
      "median:", trades_per_trader.median(), "std:", trades_per_trader.std().round(1))

campaigns_per_trader = trades_with_trader_df.groupby("traderId")["campaignId"].nunique()
print("\nTraders in >1 campaign:", (campaigns_per_trader > 1).sum(), "of", len(campaigns_per_trader))

print("\nWinner/loser split:", rebuilt_account_summary["outcome"].value_counts().to_dict())



Trades per trader — mean: 13.1 median: 5.0 std: 26.7

Traders in >1 campaign: 1313 of 3550

Winner/loser split: {'loser': 2211, 'winner': 1339}


IMP : Winners = positive net profit ; Losers = negative net profit

# EDA

In [6]:
#Test 1: Median holding time, winner vs loser
hold_by_trader = trades_with_trader_df.groupby("traderId")["durationSec"].median().reset_index(name="median_hold")
rebuilt_account_summary = rebuilt_account_summary.merge(hold_by_trader, on="traderId", how="left")
w = rebuilt_account_summary[rebuilt_account_summary["outcome"]=="winner"]["median_hold"].dropna()
l = rebuilt_account_summary[rebuilt_account_summary["outcome"]=="loser"]["median_hold"].dropna()
u1, p1 = stats.mannwhitneyu(w, l)
print(f"\n[1] Holding time — winner median: {w.median():.1f}s, loser: {l.median():.1f}s, p={p1:.4f}")



[1] Holding time — winner median: 610.0s, loser: 480.0s, p=0.0009


The results are significant, winners hold longer

In [7]:
#Test 2: Commission vs directional loss

total_comm = trades_with_trader_df["commission"].sum()
total_dir = trades_with_trader_df["profit"].sum()
print(f"[2] Commission: {total_comm:.2f}, Directional: {total_dir:.2f}, Ratio: {abs(total_comm/total_dir):.1f}x")




[2] Commission: -298472.87, Directional: -10487.16, Ratio: 28.5x


In [8]:
#Test 3: No-SL 50% threshold split

nosl_pct = trades_with_trader_df.groupby("traderId")["has_SL"].apply(lambda x: 100*(~x).mean()).reset_index(name="no_sl_pct_check")
rebuilt_account_summary = rebuilt_account_summary.merge(nosl_pct, on="traderId", how="left")
high = rebuilt_account_summary[rebuilt_account_summary["no_sl_pct_check"] >= 50]["total_netProfit"]
low = rebuilt_account_summary[rebuilt_account_summary["no_sl_pct_check"] < 50]["total_netProfit"]
t3, p3 = stats.ttest_ind(high, low, equal_var=False)
print(f"[3] No-SL split — high avg profit: {high.mean():.2f}, low: {low.mean():.2f}, p={p3:.4f}")

[3] No-SL split — high avg profit: -94.78, low: -75.95, p=0.1716


The results are not significant

In [9]:
#Test 4: Position size after loss vs after win

def get_after_loss_win_sizes(df, id_col):
    df = df.sort_values([id_col, "openDateTime"]).copy()
    df["prev_netProfit"] = df.groupby(id_col)["netProfit"].shift(1)
    df["prev_outcome"] = df["prev_netProfit"].apply(lambda x: "after_loss" if x < 0 else ("after_win" if x >= 0 else None))
    sizes = df.groupby([id_col, "prev_outcome"])["amount"].mean().unstack()
    sizes.columns = ["after_loss", "after_win"]
    return sizes.dropna()

sizes = get_after_loss_win_sizes(trades_with_trader_df, "traderId")
w4, p4 = stats.wilcoxon(sizes["after_loss"], sizes["after_win"])
print(f"[4] Size after loss: {sizes['after_loss'].median():.3f}, after win: {sizes['after_win'].median():.3f}, p={p4:.4f} (n={len(sizes)})")


[4] Size after loss: 0.159, after win: 0.150, p=0.0033 (n=2043)


The results are significant but we must tread lighty here because when this test was run on the masterdataset as accountID as the unique identifier, the results were actually the opposite of this. 

In [10]:
#Test 5: Max drawdown, winner vs loser

def fast_max_drawdown(df, id_col):
    df = df.sort_values([id_col, "openDateTime"]).copy()
    df["equity"] = df.groupby(id_col)["netProfit"].cumsum()
    df["running_max"] = df.groupby(id_col)["equity"].cummax()
    df["drawdown"] = df["equity"] - df["running_max"]
    return df.groupby(id_col)["drawdown"].min().reset_index(name="max_drawdown")

dd = fast_max_drawdown(trades_with_trader_df, "traderId")
rebuilt_account_summary = rebuilt_account_summary.merge(dd, on="traderId", how="left", suffixes=("", "_dup"))
w5 = rebuilt_account_summary[rebuilt_account_summary["outcome"]=="winner"]["max_drawdown"].dropna()
l5 = rebuilt_account_summary[rebuilt_account_summary["outcome"]=="loser"]["max_drawdown"].dropna()
u5, p5 = stats.mannwhitneyu(w5, l5)
print(f"[5] Max drawdown — winner: {w5.median():.1f}, loser: {l5.median():.1f}, p={p5:.4f}")



[5] Max drawdown — winner: -30.0, loser: -230.9, p=0.0000


Strong result, this helps seperate winners from losers 

-------------------------------------------------------------------------------------------------------------

# ADDITIONAL EDA ( WE CAN ADD CODE HERE AND CLEAN UP THIS PART LATER)

------------------------------------------------------------------------------------------------------------

## EDA FROM DEVANSHI

We want to check what the trader CROWD does in an hour and if it may predict what happens in the next hour

In [ ]:
#all the trading details aggregated wrt whats happening by the hour
flow_by_hour = con.execute("""
    SELECT 
        campaignId,
        DATE_TRUNC('hour', openDateTime) AS hour_bucket,
        SUM(CASE WHEN side='BUY' THEN amount ELSE 0 END) AS buy_volume,
        SUM(CASE WHEN side='SELL' THEN amount ELSE 0 END) AS sell_volume,
        SUM(CASE WHEN side='BUY' THEN amount ELSE -amount END) AS net_flow,
        COUNT(*) AS n_trades,
        AVG(openPrice) AS avg_price,
        SUM(CASE WHEN NOT has_SL THEN 1 ELSE 0 END) AS n_no_sl,
        AVG(netProfit) AS avg_netProfit
    FROM trades_with_trader
    GROUP BY campaignId, hour_bucket
    ORDER BY campaignId, hour_bucket
""").fetchdf()

print(flow_by_hour.shape)
print(flow_by_hour.head(10))

(755, 9)
  campaignId               hour_bucket  buy_volume  sell_volume  net_flow  \
0         33 2026-02-24 06:00:00+05:30        0.35         0.44     -0.09   
1         33 2026-02-24 07:00:00+05:30        4.78         1.81      2.97   
2         33 2026-02-24 08:00:00+05:30        1.57         5.07     -3.50   
3         33 2026-02-24 09:00:00+05:30        3.26         2.75      0.51   
4         33 2026-02-24 10:00:00+05:30        3.50         3.28      0.22   
5         33 2026-02-24 11:00:00+05:30        5.79         2.46      3.33   
6         33 2026-02-24 12:00:00+05:30        9.81         4.07      5.74   
7         33 2026-02-24 13:00:00+05:30        5.43         3.94      1.49   
8         33 2026-02-24 14:00:00+05:30        3.10         1.32      1.78   
9         33 2026-02-24 15:00:00+05:30        2.09         0.28      1.81   

   n_trades    avg_price  n_no_sl  avg_netProfit  
0        16  5186.930625     10.0       5.259375  
1        41  5171.663659     31.0      35

In [13]:
# trying to check if the info we have about a CURRENT hour helps predict what may happen in the NEXT hour

#building the next hours price change column ( how much price moved after the current hour)
flow_by_hour = flow_by_hour.sort_values(["campaignId", "hour_bucket"]).reset_index(drop=True)

flow_by_hour["next_price"] = flow_by_hour.groupby("campaignId")["avg_price"].shift(-1)
flow_by_hour["price_change_next_hour"] = flow_by_hour["next_price"] - flow_by_hour["avg_price"]

#running the lagged correlation scan 
candidate_metrics = ["net_flow", "buy_volume", "sell_volume", "n_trades", "n_no_sl", "avg_netProfit"]

correlations = flow_by_hour[candidate_metrics + ["price_change_next_hour"]].corr()["price_change_next_hour"].drop("price_change_next_hour")

print(correlations.sort_values(key=abs, ascending=False))

net_flow         0.092270
buy_volume       0.046748
n_trades         0.031362
n_no_sl          0.015785
sell_volume     -0.008563
avg_netProfit    0.005819
Name: price_change_next_hour, dtype: float64


this is a weak result

In [15]:
flow_by_hour["net_flow_pct"] = flow_by_hour["net_flow"] / (flow_by_hour["buy_volume"] + flow_by_hour["sell_volume"])
correlation_pct = flow_by_hour[["net_flow_pct", "price_change_next_hour"]].corr()
print(correlation_pct)

                        net_flow_pct  price_change_next_hour
net_flow_pct                1.000000                0.090816
price_change_next_hour      0.090816                1.000000


In [16]:
for lag in [1, 2, 3, 4]:
    flow_by_hour[f"price_change_lag{lag}"] = (
        flow_by_hour.groupby("campaignId")["avg_price"].shift(-lag) - flow_by_hour["avg_price"]
    )
    corr = flow_by_hour["net_flow"].corr(flow_by_hour[f"price_change_lag{lag}"])
    print(f"Lag {lag} hour(s): correlation = {corr:.4f}")

Lag 1 hour(s): correlation = 0.0923
Lag 2 hour(s): correlation = 0.0485
Lag 3 hour(s): correlation = 0.0285
Lag 4 hour(s): correlation = 0.0091


this is obvious, the weak correlation keeps becoming less and less as time passes

In [17]:
extreme_buy = flow_by_hour[flow_by_hour["net_flow"] > flow_by_hour["net_flow"].quantile(0.9)]
extreme_sell = flow_by_hour[flow_by_hour["net_flow"] < flow_by_hour["net_flow"].quantile(0.1)]

print("After extreme net-buying hours, avg next-hour price change:", extreme_buy["price_change_next_hour"].mean())
print("After extreme net-selling hours, avg next-hour price change:", extreme_sell["price_change_next_hour"].mean())

After extreme net-buying hours, avg next-hour price change: 2.0880320037302713
After extreme net-selling hours, avg next-hour price change: -2.945024260839857


This may be a obvious result but gold prices moved a lot over 4.5 months(during all the campaigns) so we need to check if extreme buying predicts a move beyond what the general trend would already predict.Simplest version: detrend by looking at the price change relative to the campaign's own average movement that day, not the raw price change.

In [18]:
# subtract each campaign's own average hourly price change, to remove the trend
flow_by_hour["campaign_avg_change"] = flow_by_hour.groupby("campaignId")["price_change_next_hour"].transform("mean")
flow_by_hour["detrended_change"] = flow_by_hour["price_change_next_hour"] - flow_by_hour["campaign_avg_change"]

extreme_buy = flow_by_hour[flow_by_hour["net_flow_pct"] > flow_by_hour["net_flow_pct"].quantile(0.9)]
extreme_sell = flow_by_hour[flow_by_hour["net_flow_pct"] < flow_by_hour["net_flow_pct"].quantile(0.1)]

print("Extreme buy, detrended next-hour change:", extreme_buy["detrended_change"].mean())
print("Extreme sell, detrended next-hour change:", extreme_sell["detrended_change"].mean())

Extreme buy, detrended next-hour change: 1.9329855989516627
Extreme sell, detrended next-hour change: -2.379112477101045


In [23]:
flow_by_hour = flow_by_hour.sort_values(["campaignId", "hour_bucket"]).reset_index(drop=True)

behavior_cols = ["net_flow_pct", "n_trades", "n_no_sl", "avg_netProfit", "buy_volume", "sell_volume"]

for col in behavior_cols:
    flow_by_hour[f"next_{col}"] = flow_by_hour.groupby("campaignId")[col].shift(-1)

# also useful: no-SL as a RATE, not raw count, so it's comparable across busier/quieter hours
flow_by_hour["no_sl_rate"] = flow_by_hour["n_no_sl"] / flow_by_hour["n_trades"]
flow_by_hour["next_no_sl_rate"] = flow_by_hour.groupby("campaignId")["no_sl_rate"].shift(-1)

this_hour = ["net_flow_pct", "n_trades", "no_sl_rate", "avg_netProfit"]
next_hour = ["next_net_flow_pct", "next_n_trades", "next_no_sl_rate", "next_avg_netProfit"]

corr_matrix = flow_by_hour[this_hour + next_hour].corr().loc[this_hour, next_hour]
print(corr_matrix.round(3))



               next_net_flow_pct  next_n_trades  next_no_sl_rate  \
net_flow_pct               0.133         -0.062           -0.012   
n_trades                  -0.077          0.763            0.039   
no_sl_rate                 0.046          0.072            0.234   
avg_netProfit             -0.030          0.050            0.012   

               next_avg_netProfit  
net_flow_pct               -0.080  
n_trades                    0.029  
no_sl_rate                  0.084  
avg_netProfit               0.008  


1. this just shows that high number of trades are just followed by high number of trades in the next hour (0.763) 
2. but also that no sl is maybe followed by no sl in the next hour 

lets look at the no sl thing 

In [32]:
# among NO-SL trades only, look at the distribution of losses
no_sl_losses = trades_with_trader_df[
    (~trades_with_trader_df["has_SL"]) & (trades_with_trader_df["netProfit"] < 0)
]["netProfit"]

print(no_sl_losses.describe())
print("\nWhat % of no-SL losses are 'small/disciplined' (say, better than -50) vs 'catastrophic' (worse than -300)?")
print("Disciplined-ish:", (no_sl_losses > -50).mean())
print("Catastrophic:", (no_sl_losses < -300).mean())

count    11323.000000
mean       -91.286328
std        103.795039
min       -944.100000
25%       -135.600000
50%        -51.960000
75%        -14.105000
max         -0.010000
Name: netProfit, dtype: float64

What % of no-SL losses are 'small/disciplined' (say, better than -50) vs 'catastrophic' (worse than -300)?
Disciplined-ish: 0.48988783891194915
Catastrophic: 0.04936854190585534


In [33]:
catastrophic = trades_with_trader_df[
    (~trades_with_trader_df["has_SL"]) & (trades_with_trader_df["netProfit"] < -300)
]

disciplined = trades_with_trader_df[
    (~trades_with_trader_df["has_SL"]) & (trades_with_trader_df["netProfit"] < 0) & (trades_with_trader_df["netProfit"] > -50)
]

print("Catastrophic trades — median duration:", catastrophic["durationSec"].median(), 
      "median size:", catastrophic["amount"].median())
print("Disciplined trades — median duration:", disciplined["durationSec"].median(), 
      "median size:", disciplined["amount"].median())

Catastrophic trades — median duration: 1267.0 median size: 0.3
Disciplined trades — median duration: 63.0 median size: 0.1


In [34]:
no_sl_trades = trades_with_trader_df[~trades_with_trader_df["has_SL"]].copy()
no_sl_trades["is_catastrophic"] = no_sl_trades["netProfit"] < -300

# check: among trades that are STILL OPEN past 5 minutes (300 sec) with above-median size,
# what fraction eventually go catastrophic, vs trades that don't meet that criteria?
median_size = no_sl_trades["amount"].median()

no_sl_trades["flagged"] = (no_sl_trades["durationSec"] > 300) & (no_sl_trades["amount"] > median_size)

print(no_sl_trades.groupby("flagged")["is_catastrophic"].mean())
print(no_sl_trades["flagged"].value_counts())

flagged
False    0.015265
True     0.098732
Name: is_catastrophic, dtype: float64
flagged
False    22338
True      2208
Name: count, dtype: int64


In [35]:
catastrophic = trades_with_trader_df[
    (~trades_with_trader_df["has_SL"]) & (trades_with_trader_df["netProfit"] < -300)
]

disciplined = trades_with_trader_df[
    (~trades_with_trader_df["has_SL"]) & (trades_with_trader_df["netProfit"] < 0) & (trades_with_trader_df["netProfit"] > -50)
]

print("Catastrophic trades — avg reverseProfit:", catastrophic["reverseProfit"].mean())
print("Disciplined trades — avg reverseProfit:", disciplined["reverseProfit"].mean())

# also worth checking: reverseProfit specifically for the FLAGGED group vs non-flagged
print("\nFlagged trades — avg reverseProfit:", no_sl_trades[no_sl_trades["flagged"]]["reverseProfit"].mean())
print("Non-flagged trades — avg reverseProfit:", no_sl_trades[~no_sl_trades["flagged"]]["reverseProfit"].mean())

Catastrophic trades — avg reverseProfit: 380.3320035778176
Disciplined trades — avg reverseProfit: 10.441956913647017

Flagged trades — avg reverseProfit: 23.03688405797101
Non-flagged trades — avg reverseProfit: -1.1453796221685022
